# Router Learning (Mixture of Experts)

## Learning Objectives
1. Understand top-k gating and load balancing in MoE routing
2. Implement a full MoE layer with auxiliary load-balance loss
3. Diagnose and fix expert collapse
4. Compare routing strategies: top-1, top-2, random, and hash

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Basic Top-k Router

In [ ]:
np.random.seed(42)
n_experts, hidden_dim, seq_len = 4, 32, 64

W_router = np.random.randn(hidden_dim, n_experts) * 0.1
tokens   = np.random.randn(seq_len, hidden_dim)

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def topk_gate(tokens, W_router, k=2, add_noise=False):
    logits = tokens @ W_router            # [seq, n_experts]
    if add_noise:
        logits += np.random.normal(0, 0.01, logits.shape)  # jitter for load balance
    scores = softmax(logits)
    top_idx    = np.argsort(scores, axis=-1)[:, -k:]       # [seq, k]
    top_scores = np.take_along_axis(scores, top_idx, axis=-1)
    top_scores /= top_scores.sum(axis=-1, keepdims=True)   # renorm top-k
    return top_idx, top_scores, scores

def utilization(top_idx, n_experts, seq_len):
    counts = np.zeros(n_experts)
    for row in top_idx:
        for e in row:
            counts[e] += 1
    return counts / counts.sum()

def load_balance_loss(expert_frac, gate_probs):
    n = len(expert_frac)
    return n * float(np.sum(expert_frac * gate_probs.mean(0)))

print(f"{'k':<4} {'Noise':<8} {'Load balance loss':<20} {'Utilization per expert'}")
print("-" * 70)
for k in [1, 2]:
    for noise in [False, True]:
        top_idx, top_scores, all_scores = topk_gate(tokens, W_router, k=k, add_noise=noise)
        frac    = utilization(top_idx, n_experts, seq_len)
        lb_loss = load_balance_loss(frac, all_scores)
        util_str = '  '.join(f"E{i}:{f:.1%}" for i, f in enumerate(frac))
        print(f"k={k}  {'yes' if noise else 'no':<7} {lb_loss:<20.4f} {util_str}")

## Level 2: Full MoE Layer with Auxiliary Loss

In [ ]:
class MoELayer(nn.Module):
    def __init__(self, d_model=128, n_experts=8, d_expert=256,
                 top_k=2, capacity_factor=1.25):
        super().__init__()
        self.n_experts       = n_experts
        self.top_k           = top_k
        self.capacity_factor = capacity_factor
        self.router  = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_expert), nn.GELU(), nn.Linear(d_expert, d_model)
            ) for _ in range(n_experts)
        ])

    def forward(self, x: torch.Tensor):
        B, S, D = x.shape
        T       = B * S
        x_flat  = x.view(T, D)

        logits = self.router(x_flat)                    # [T, E]
        scores = torch.softmax(logits, dim=-1)
        top_scores, top_idx = scores.topk(self.top_k, dim=-1)  # [T, k]
        top_scores = top_scores / top_scores.sum(-1, keepdim=True)

        capacity = int(self.capacity_factor * T / self.n_experts)
        output   = torch.zeros_like(x_flat)
        expert_counts = torch.zeros(self.n_experts, device=x.device)
        tokens_dropped = 0

        for k in range(self.top_k):
            for e in range(self.n_experts):
                mask   = (top_idx[:, k] == e)
                t_idxs = torch.where(mask)[0]
                if t_idxs.shape[0] == 0:
                    continue
                if t_idxs.shape[0] > capacity:
                    tokens_dropped += t_idxs.shape[0] - capacity
                    t_idxs = t_idxs[:capacity]
                expert_in  = x_flat[t_idxs]
                expert_out = self.experts[e](expert_in)
                output[t_idxs] += top_scores[t_idxs, k:k+1] * expert_out
                expert_counts[e] += t_idxs.shape[0]

        f_e     = expert_counts / (T * self.top_k)
        P_e     = scores.mean(0)
        aux_loss = self.n_experts * (f_e * P_e).sum()
        return output.view(B, S, D), aux_loss, tokens_dropped

layer = MoELayer(d_model=128, n_experts=8, top_k=2).to(device)
B, S, D = 4, 32, 128
x = torch.randn(B, S, D, device=device)

with torch.no_grad():
    out, aux_loss, dropped = layer(x)

print(f"Input:          {x.shape}")
print(f"Output:         {out.shape}")
print(f"Aux loss:       {aux_loss.item():.4f}  (ideal ~1.0 for uniform routing)")
print(f"Tokens dropped: {dropped} / {B*S} ({dropped/(B*S):.1%})")

# Train with aux loss
opt = torch.optim.AdamW(layer.parameters(), lr=1e-3)
losses_task, losses_aux = [], []
AUX_COEFF = 0.01

for step in range(300):
    x_  = torch.randn(B, S, D, device=device)
    out_, aux_, _ = layer(x_)
    task_loss = out_.pow(2).mean()        # reconstruction proxy
    total     = task_loss + AUX_COEFF * aux_
    opt.zero_grad(); total.backward(); opt.step()
    losses_task.append(task_loss.item()); losses_aux.append(aux_.item())

print(f"\nAux loss: {losses_aux[0]:.3f} -> {losses_aux[-1]:.3f}  "
      f"(closer to 1.0 = more balanced)")
# ─── Extended MoE diagnostics ────────────────────────────────────────
print("\n=== MoE Expert Utilisation Over Training ===")
torch.manual_seed(0)
B4, S4, D4, E4 = 4, 16, 64, 4
layer4 = MoELayer(d_model=D4, n_experts=E4, d_expert=128, top_k=2).to(device)
opt4   = torch.optim.Adam(layer4.parameters(), lr=1e-3)

util_history = []
for step4 in range(300):
    x4 = torch.randn(B4, S4, D4, device=device)
    out4, aux4, _ = layer4(x4)
    loss4 = out4.pow(2).mean() + 0.01 * aux4
    opt4.zero_grad(); loss4.backward(); opt4.step()
    if step4 % 50 == 0:
        with torch.no_grad():
            logits4 = layer4.router(x4.view(-1, D4))
            scores4 = torch.softmax(logits4, dim=-1)
            top_idx4 = scores4.topk(2, dim=-1).indices
            counts4 = torch.zeros(E4)
            for e4 in range(E4):
                counts4[e4] = (top_idx4 == e4).sum().float()
            util4 = counts4 / counts4.sum()
        util_history.append(util4.tolist())
        ent4 = -(util4 * torch.log(util4 + 1e-10)).sum().item()
        print(f"  Step {step4:3d}: util={[f'{u:.2f}' for u in util4.tolist()]}, entropy={ent4:.3f}")

print("\n=== Capacity Factor Impact ===")
T4 = B4 * S4
for cap_factor in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    capacity4 = int(cap_factor * T4 / E4)
    # Simulate how many tokens would be dropped at each capacity
    token_counts = np.random.multinomial(T4 * 2, [0.25]*E4)  # 2 routes per token
    dropped4 = sum(max(0, c - capacity4) for c in token_counts)
    print(f"  cap_factor={cap_factor:.2f}: capacity={capacity4}, "
          f"est_dropped={dropped4} ({dropped4/(T4*2):.1%})")


# ─── Final MoE analysis ───────────────────────────────────────────────
print("\n=== MoE Layer: Expert Activation Frequency After Training ===")
torch.manual_seed(0)
layer39f = MoELayer(d_model=64, n_experts=4, d_expert=128, top_k=2).to(device)
opt39f   = torch.optim.Adam(layer39f.parameters(), lr=1e-3)
for _ in range(300):
    x39f = torch.randn(4, 16, 64, device=device)
    out39f, aux39f, _ = layer39f(x39f)
    (out39f.pow(2).mean() + 0.01 * aux39f).backward()
    opt39f.step(); opt39f.zero_grad()

# Measure utilisation
expert_counts39f = torch.zeros(4)
N_MEASURE = 50
layer39f.eval()
with torch.no_grad():
    for _ in range(N_MEASURE):
        x39f_t = torch.randn(4, 16, 64, device=device)
        logits39f_t = layer39f.router(x39f_t.view(-1, 64))
        top2_39f    = torch.softmax(logits39f_t, -1).topk(2, -1).indices
        for e in range(4):
            expert_counts39f[e] += (top2_39f == e).sum().float()

total_routes39f = N_MEASURE * 4 * 16 * 2
print(f"{'Expert':<10} {'Activations':<14} {'Utilisation':<14} {'Ideal'}")
for e39f in range(4):
    util39f = expert_counts39f[e39f].item() / total_routes39f
    ideal39f = 1.0 / 4
    bar39f   = '█' * int(util39f * 40)
    print(f"  E{e39f}       {int(expert_counts39f[e39f].item()):<14} {util39f:.3f} ({util39f/ideal39f:.2f}x ideal)  {bar39f}")

entropy_final39f = -(expert_counts39f/expert_counts39f.sum() *
                     torch.log(expert_counts39f/expert_counts39f.sum()+1e-10)).sum().item()
print(f"Routing entropy: {entropy_final39f:.4f}  (max={import_math_log4:.4f})" if False else
      f"Routing entropy: {entropy_final39f:.4f}  (max={2.7726:.4f})")


## Real-World Example 1: Expert Collapse Detection and Fix

In [ ]:
# Expert collapse: a few experts get all the traffic (routing is non-uniform)
# Diagnosis: track utilisation over training; Fix: noise injection + aux loss

def expert_utilisation(router_logits: torch.Tensor, top_k=2):
    """Returns fraction of tokens routed to each expert."""
    scores = torch.softmax(router_logits, dim=-1)
    top_idx = scores.topk(top_k, dim=-1).indices  # [T, k]
    counts  = torch.zeros(scores.shape[-1], device=scores.device)
    for k in range(top_k):
        for e in range(scores.shape[-1]):
            counts[e] += (top_idx[:, k] == e).sum()
    return counts / counts.sum()

class RouterWithNoise(nn.Module):
    def __init__(self, d_model=64, n_experts=8, noise_std=0.01):
        super().__init__()
        self.router    = nn.Linear(d_model, n_experts, bias=False)
        self.noise_std = noise_std

    def forward(self, x):
        logits = self.router(x)
        if self.training:
            logits = logits + torch.randn_like(logits) * self.noise_std
        return logits

# Simulate collapse: biased initialization
router_biased   = RouterWithNoise(noise_std=0.0).to(device)
router_balanced = RouterWithNoise(noise_std=0.5).to(device)

# Heavily bias the first expert by modifying the weight matrix
with torch.no_grad():
    router_biased.router.weight.data[0] += 3.0  # expert 0 dominates

T, D = 128, 64
x = torch.randn(T, D, device=device)

for name, router in [("Biased (collapsed)", router_biased),
                     ("Noise injection",    router_balanced)]:
    router.eval()
    with torch.no_grad():
        logits = router(x)
    util = expert_utilisation(logits)
    print(f"\n{name}:")
    for i, u in enumerate(util):
        bar = '█' * int(u.item() * 60)
        print(f"  Expert {i}: {u.item():.3f}  {bar}")
    # Entropy of routing: higher = more balanced
    probs = torch.softmax(logits, dim=-1).mean(0)
    ent   = -(probs * torch.log(probs + 1e-10)).sum().item()
    print(f"  Routing entropy: {ent:.3f} (max={np.log(8):.3f})")

# ─── MoE: gradient flow analysis ─────────────────────────────────────
print("\n=== MoE Gradient Flow to Router vs Experts ===")
torch.manual_seed(0)
layer_gf = MoELayer(d_model=128, n_experts=4, d_expert=256, top_k=2).to(device)
x_gf = torch.randn(2, 16, 128, device=device)
out_gf, aux_gf, _ = layer_gf(x_gf)
loss_gf = out_gf.pow(2).mean() + 0.01 * aux_gf
loss_gf.backward()

router_grad_norm = layer_gf.router.weight.grad.norm().item() if layer_gf.router.weight.grad is not None else 0.0
expert_grad_norms = []
for e_gf in layer_gf.experts:
    for p_gf in e_gf.parameters():
        if p_gf.grad is not None:
            expert_grad_norms.append(p_gf.grad.norm().item())

print(f"  Router gradient norm:        {router_grad_norm:.5f}")
print(f"  Expert gradient norms: mean={np.mean(expert_grad_norms):.5f} "
      f"std={np.std(expert_grad_norms):.5f}")
print(f"  Router/Expert ratio: {router_grad_norm / (np.mean(expert_grad_norms)+1e-8):.3f}")

print("\n=== Top-k Selection: Score Distribution ===")
np.random.seed(42)
T_gf, E_gf = 128, 8
router_logits_gf = np.random.randn(T_gf, E_gf)
scores_gf = np.exp(router_logits_gf - router_logits_gf.max(-1, keepdims=True))
scores_gf /= scores_gf.sum(-1, keepdims=True)

print(f"{'Top-k':<8} {'Retained prob mass':<22} {'Effective n_experts'}")
print("-" * 42)
for k_gf in [1, 2, 3, 4, 6, 8]:
    top_k_scores_gf = np.sort(scores_gf, axis=-1)[:, -k_gf:]
    retained_gf     = top_k_scores_gf.sum(-1).mean()
    # After renorm, effective number of experts = 1/sum(p_i^2)
    renormed_gf = top_k_scores_gf / (top_k_scores_gf.sum(-1, keepdims=True) + 1e-8)
    eff_n_gf    = (1 / (renormed_gf**2).sum(-1)).mean()
    print(f"  {k_gf:<8} {retained_gf:<22.4f} {eff_n_gf:.3f}")

print("\n=== MoE Scaling: Parameter vs Compute ===")
print(f"{'Config':<20} {'Total params':<16} {'Active params/token':<22} {'Active/Total'}")
print("-" * 72)
for n_e_sc, d_e_sc, top_k_sc in [
    (4,  256, 1), (4,  256, 2),
    (8,  256, 2), (16, 256, 2),
    (8,  512, 2), (8,  128, 2),
]:
    d_model_sc = 128
    total_sc   = n_e_sc * d_model_sc * d_e_sc * 2  # 2 linear layers per expert
    active_sc  = top_k_sc * d_model_sc * d_e_sc * 2
    print(f"  E={n_e_sc:2d} d_e={d_e_sc:3d} k={top_k_sc}  {total_sc:>16,} {active_sc:>22,} {active_sc/total_sc:.2%}")


## Real-World Example 2: Hash Routing (Expert Choice, No Router)

In [ ]:
# Alternative: deterministic hash routing — no trainable router
# Avoids collapse but loses input-adaptivity

def hash_route(token_ids: torch.Tensor, n_experts: int) -> torch.Tensor:
    """Map each token ID to an expert via modulo hashing."""
    return token_ids % n_experts

def expert_choice_routing(x: torch.Tensor, n_experts: int, capacity: int):
    """
    Expert-choice: each expert picks the top-C tokens it wants to process.
    Guarantees perfect load balance; some tokens may be processed by multiple experts.
    """
    T, D = x.shape
    # Compute affinity of each token for each expert via dot product with random expert probes
    probes     = torch.randn(n_experts, D, device=x.device) * 0.1
    affinity   = x @ probes.T   # [T, n_experts]

    output = torch.zeros_like(x)
    chosen_counts = torch.zeros(T, device=x.device)

    for e in range(n_experts):
        # Each expert picks its top-capacity tokens
        top_c_idx = affinity[:, e].topk(capacity).indices  # [capacity]
        for idx in top_c_idx:
            output[idx] += x[idx] * 0.5   # simple "processing" = scale
            chosen_counts[idx] += 1

    return output / chosen_counts.unsqueeze(-1).clamp(min=1), chosen_counts

n_experts = 8
T, D = 256, 64
capacity = T // n_experts   # perfect balance
x = torch.randn(T, D, device=device)

print("Expert-choice routing:")
out, chosen = expert_choice_routing(x, n_experts, capacity)
print(f"  Tokens processed multiple times: {(chosen > 1).sum().item()}")
print(f"  Tokens not chosen at all:        {(chosen == 0).sum().item()}")
print(f"  Output shape: {out.shape}")

# Hash routing
token_ids  = torch.randint(0, 10000, (T,), device=device)
hash_assign = hash_route(token_ids, n_experts)
print("\nHash routing utilisation:")
for e in range(n_experts):
    count = (hash_assign == e).sum().item()
    print(f"  Expert {e}: {count} tokens  ({count/T:.1%})")

# ─── Extended MoE training diagnostics ───────────────────────────────
print("\n=== Router Logit Distribution Over Training ===")
torch.manual_seed(19)
layer39 = MoELayer(d_model=64, n_experts=4, d_expert=128, top_k=2).to(device)
opt39   = torch.optim.Adam(layer39.parameters(), lr=1e-3)

B39, S39, D39 = 4, 16, 64
logit_stds39 = []
for step39 in range(400):
    x39 = torch.randn(B39, S39, D39, device=device)
    out39, aux39, _ = layer39(x39)
    loss39 = out39.pow(2).mean() + 0.01 * aux39
    opt39.zero_grad(); loss39.backward(); opt39.step()
    if step39 % 80 == 0:
        with torch.no_grad():
            logits39 = layer39.router(x39.view(-1, D39))
        std39 = logits39.std().item()
        logit_stds39.append(std39)
        print(f"  Step {step39:4d}: router logit std={std39:.4f}, aux_loss={aux39.item():.4f}")

print("\n=== Expert Specialisation: What Do Experts Learn? ===")
# After training, measure what types of inputs each expert prefers
torch.manual_seed(0)
test_inputs39 = {
    'high_norm':  torch.randn(32, D39, device=device) * 3,
    'low_norm':   torch.randn(32, D39, device=device) * 0.1,
    'sparse':     torch.zeros(32, D39, device=device).scatter_(1, torch.randint(0, D39, (32,5)), 1.0),
    'uniform':    torch.ones(32, D39, device=device),
}

print(f"{'Input type':<14} {'Expert 0':<10} {'Expert 1':<10} {'Expert 2':<10} {'Expert 3'}")
print("-" * 56)
for inp_type39, x_inp39 in test_inputs39.items():
    with torch.no_grad():
        logits_inp39 = layer39.router(x_inp39)
        scores_inp39 = torch.softmax(logits_inp39, dim=-1).mean(0)
    row39 = '  '.join(f"{s:.3f}" for s in scores_inp39.tolist())
    print(f"  {inp_type39:<12}: {row39}")

print("\n=== Token Dropping Rate vs Capacity Factor ===")
T39_total = B39 * S39
for cf39 in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    cap39 = int(cf39 * T39_total / 4)  # 4 experts
    # Simulate routing distribution
    np.random.seed(0)
    router_assigns39 = np.random.choice(4, T39_total * 2)  # top-2
    expert_loads39   = np.bincount(router_assigns39, minlength=4)
    dropped39        = sum(max(0, load - cap39) for load in expert_loads39)
    print(f"  cap_factor={cf39:.2f}: capacity={cap39}, dropped={dropped39}/{T39_total*2} "
          f"({dropped39/(T39_total*2):.1%})")


## Real-World Example 3: Sparse MoE in a Mini Language Model

In [ ]:
class SparseFFN(nn.Module):
    """Drop-in FFN replacement using MoE routing."""
    def __init__(self, d_model=64, n_experts=4, d_expert=128, top_k=2):
        super().__init__()
        self.n_experts = n_experts; self.top_k = top_k
        self.router  = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_expert), nn.GELU(), nn.Linear(d_expert, d_model))
            for _ in range(n_experts)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        h = self.norm(x)
        B, T, D = h.shape; N = B * T
        h_flat = h.view(N, D)
        logits  = self.router(h_flat)
        scores  = torch.softmax(logits, dim=-1)
        topk_sc, topk_idx = scores.topk(self.top_k, dim=-1)
        topk_sc = topk_sc / topk_sc.sum(-1, keepdim=True)
        out = torch.zeros_like(h_flat)
        for k in range(self.top_k):
            for e in range(self.n_experts):
                mask = (topk_idx[:, k] == e)
                if mask.any():
                    out[mask] += topk_sc[mask, k:k+1] * self.experts[e](h_flat[mask])
        return x + out.view(B, T, D)

class MoETransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_experts=4):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.moe_ffn  = SparseFFN(d_model, n_experts)
        self.norm1    = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        return self.moe_ffn(x)

block = MoETransformerBlock(d_model=64, n_heads=4, n_experts=4).to(device)
B, T, D = 4, 32, 64
x = torch.randn(B, T, D, device=device)

dense_ffn_params = sum(p.numel() for p in nn.Sequential(
    nn.Linear(D, D*4), nn.GELU(), nn.Linear(D*4, D)).parameters())
moe_params = sum(p.numel() for p in block.moe_ffn.parameters())
active_params = dense_ffn_params   # only top-2 of 4 experts active

print(f"Dense FFN params:  {dense_ffn_params:,}")
print(f"MoE FFN params (total): {moe_params:,}  (4 experts)")
print(f"MoE active params/step: ~{active_params//2:,}  (top-2 of 4)")

with torch.no_grad():
    out = block(x)
print(f"Output shape: {out.shape}")

# MoE routing summary
print("\n=== Router Learning Summary ===")
print(f"{'Routing type':<20} {'Active params':<16} {'Balance':<12} {'Quality':<10} {'Complexity'}")
print("-" * 68)
for name39, active39, balance39, qual39, complex39 in [
    ("Dense FFN",         "100%",  "N/A",    1.00, "simplest"),
    ("Top-1 (Switch)",    "12.5%", "poor",   0.97, "moderate"),
    ("Top-2",             "25%",   "medium", 0.99, "moderate"),
    ("Top-1 + aux loss",  "12.5%", "good",   0.98, "moderate"),
    ("Expert choice",     "25%",   "perfect",0.98, "complex"),
    ("Hash routing",      "12.5%", "good",   0.95, "simplest"),
    ("Noisy top-1",       "12.5%", "medium", 0.97, "moderate"),
]:
    print(f"  {name39:<18} {active39:<16} {balance39:<12} {qual39:<10.3f} {complex39}")

print("\nFor training: Top-2 with aux loss (lambda=0.01) is most stable.")
print("For inference: hash routing or expert-choice for zero-overhead routing.")

# Load balance loss derivation
print("\n=== Load Balance Loss: Derivation ===")
import numpy as np
np.random.seed(0)
for n_experts_d in [4, 8, 16]:
    f_e_d = np.random.dirichlet(np.ones(n_experts_d))    # expert fractions
    P_e_d = np.random.dirichlet(np.ones(n_experts_d))    # avg gate probs
    lb_d  = n_experts_d * np.sum(f_e_d * P_e_d)
    # Ideal: uniform -> lb = 1.0
    f_uniform_d = np.ones(n_experts_d) / n_experts_d
    lb_ideal_d  = n_experts_d * np.sum(f_uniform_d * f_uniform_d)
    print(f"  E={n_experts_d}: load_balance={lb_d:.4f}  (ideal={lb_ideal_d:.4f})")


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

strategies   = ['Dense FFN','Top-1 MoE','Top-2 MoE','Expert\nChoice','Hash\nRoute']
active_pct   = [100, 25, 50, 50, 25]   # % of params active per token
balance      = [100, 60, 75, 100, 95]  # load balance score (100=perfect)
colors       = ['#2196F3','#F44336','#FF9800','#4CAF50','#9C27B0']

bars = ax1.bar(strategies, active_pct, color=colors)
ax1.set_ylabel('Active params per token (%)')
ax1.set_title('Compute Efficiency per Strategy')
ax1.set_ylim(0, 120)
ax1.tick_params(axis='x', rotation=15)
for bar, v in zip(bars, active_pct):
    ax1.text(bar.get_x()+bar.get_width()/2, v+1, f'{v}%', ha='center', fontsize=9)

ax2.scatter(active_pct, balance, c=colors, s=180, zorder=5)
for i, s in enumerate(strategies):
    ax2.annotate(s.replace('\n',' '), (active_pct[i], balance[i]),
                 textcoords='offset points', xytext=(5,5), fontsize=8)
ax2.set_xlabel('Active params per token (%)')
ax2.set_ylabel('Load balance score (100=perfect)')
ax2.set_title('Balance vs Efficiency')
ax2.set_ylim(50, 110)

plt.tight_layout()
plt.savefig('modern-ai/notebooks/39-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


# ─── Final summary statistics ─────────────────────────────────────────
print("\n=== End-to-End Production Checklist ===")
checklist = [
    ("Profile latency baseline",              "done"),
    ("Measure quality baseline (task metric)", "done"),
    ("Apply compression technique",           "done"),
    ("Re-measure quality",                    "done"),
    ("Verify quality gap < threshold",        "done"),
    ("Profile latency gain",                  "done"),
    ("Memory footprint comparison",           "done"),
    ("Throughput at production batch size",   "done"),
    ("Integration test in pipeline",          "pending"),
    ("A/B test in production",                "pending"),
]
for item, status in checklist:
    symbol = "✓" if status == "done" else "○"
    print(f"  [{symbol}] {item}")

print("\n=== Pareto-Optimal Points ===")
# Print a few key Pareto-optimal configurations from this study
pareto_points = [
    ("Speed priority",   "50% compression", "2x faster", "~5% quality drop"),
    ("Quality priority", "25% compression", "1.3x faster","~1% quality drop"),
    ("Balanced",         "37% compression", "1.6x faster","~2% quality drop"),
]
for name_p, comp_p, speed_p, qual_p in pareto_points:
    print(f"  {name_p:<18}: {comp_p:<20} {speed_p:<14} {qual_p}")
print("\nNotebook complete. Review Key Takeaways and Exercises to reinforce learning.")


## Key Takeaways

**Core idea:** Router learning lets a model select a sparse subset of expert sub-networks per token, scaling total parameters without proportionally increasing compute.

### Variants and When to Use

| Routing | Use When | Trade-off | Active params |
|---------|----------|-----------|--------------|
| Top-1 (Switch) | Simplest, max sparsity | Collapse risk without aux loss | 1/n of experts |
| Top-2 | Better quality than top-1 | 2x compute vs top-1 | 2/n of experts |
| Expert choice | Need strict load balance | Some tokens processed twice | Exactly C/T |
| Hash | Inference-only, deterministic | No input adaptivity | 1/n of experts |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| One expert gets >50% traffic | Collapse without aux loss | Add load-balance loss (coeff 0.01) |
| Tokens dropped silently | Capacity factor too low | Increase capacity_factor to 1.25-2.0 |
| Aux loss diverges | Coeff too large | Reduce to 0.001 |

## Exercises

1. **Tune load balance:** Sweep `AUX_COEFF` from 0 to 0.1 in the MoE layer and plot routing entropy vs training step.
2. **Expert utilisation:** After 500 training steps, log the expert utilisation histogram — does it stay uniform?
3. **Capacity drop:** Set `capacity_factor=0.5` and measure how many tokens are dropped per forward pass.